# 02 · Models

All models are evaluated on the newest patch (held out by time).

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("LOLWP_DATA_DIR", ROOT / "data"))
ART = Path(os.environ.get("LOLWP_ARTIFACTS_DIR", ROOT / "artifacts"))
sns.set_theme(style="whitegrid")
import joblib
from sklearn.calibration import calibration_curve
from lolwp.config import STATE_FEATURES
from lolwp.features.draft import DRAFT_FEATURES
metrics = json.load(open(ART / "metrics.json"))
draft = pd.read_parquet(DATA / "draft.parquet"); state = pd.read_parquet(DATA / "state.parquet")
lr = {n: joblib.load(ART / f"{n}_model.joblib") for n in ["draft", "state", "combined"]}
pd.DataFrame(metrics["models"]).T.round(4)

**Interpretation (write 3–5 sentences after running on the real data):** _Does the draft model beat the baseline on log loss? How much does adding the draft to game state help?_

*Draft — rewrite in your own words.* The draft model only just beats the baseline: log loss 0.6923 vs 0.6924, AUC 0.529 vs 0.524, both close to a coin flip. Adding the draft to the 15:00 game state raises AUC by 0.002 (0.821 → 0.823), so at 15 minutes the draft adds almost nothing the game state doesn't already show.

## Calibration of the logistic regression models

In [ ]:
d_te = draft[draft.split == "test"]; s_te = state[state.split == "test"]
logit = pd.Series(lr["draft"].decision_function(d_te[DRAFT_FEATURES]), index=d_te.match_id)
c_te = s_te.assign(draft_logit=s_te.match_id.map(logit).to_numpy())
sets = {"draft": (d_te[DRAFT_FEATURES], d_te.blue_win), "state": (s_te[STATE_FEATURES], s_te.blue_win),
        "combined": (c_te[["draft_logit", *STATE_FEATURES]], c_te.blue_win)}
fig, ax = plt.subplots(figsize=(5, 5))
for name, (X, y) in sets.items():
    frac, mean = calibration_curve(y, lr[name].predict_proba(X)[:, 1], n_bins=10)
    ax.plot(mean, frac, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--"); ax.set(xlabel="Predicted", ylabel="Observed"); ax.legend()
plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _Are predicted probabilities trustworthy as probabilities? Where do the curves bend away from the diagonal?_

*Draft — rewrite in your own words.* The state and combined models are well calibrated: in every 10% bin, the actual Blue win rate is within 1–2 points of the predicted one. The draft model almost never leaves 40–60%, and its few edge bins (85 and 38 games) are too small to trust.

## Coefficients (standardized)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, lr):
    m = lr[name]
    pd.Series(m[-1].coef_[0], index=m.feature_names_in_).sort_values().plot.barh(ax=ax, title=name)
plt.tight_layout(); plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _Which features carry the most weight per standard deviation? Any surprising signs?_

*Draft — rewrite in your own words.* Gold dominates (1.11 log-odds per standard deviation), followed by dragons (0.40) and total level (0.27). Towers, plates, kills and the first-objective flags get near-zero or slightly negative weights. That isn't because they don't matter: their value is already in the gold lead they produced.

## Drift check

In [ ]:
pd.DataFrame(metrics["prev_patch_auc"]).T

**Interpretation (write 3–5 sentences after running on the real data):** _Does a model trained on older patches lose AUC on the newest patch compared with the previous one?_

*Draft — rewrite in your own words.* Yes for the draft: trained on older patches, it drops from 0.534 AUC on 16.20 to 0.515 on 16.21, so champion and matchup win rates go stale within one patch. The game-state model barely moves (0.825 → 0.820), and neither does the combined one (0.825 → 0.821).

## Why LightGBM did or didn't win

In [ ]:
m = pd.DataFrame(metrics["models"]).T
pd.DataFrame({k: m.loc[f"{k}_lgbm", ["auc", "log_loss"]] - m.loc[f"{k}_lr", ["auc", "log_loss"]]
              for k in ["draft", "state", "combined"]}).T

**Interpretation (write 3–5 sentences after running on the real data):** _Is any LGBM AUC gain above 0.01? Relate it to the features being mostly additive differences._

*Draft — rewrite in your own words.* No LightGBM gain clears 0.01; it trails logistic regression by 0.004–0.015 in every mode. The features are Blue-minus-Red differences whose effect is mostly additive and monotone, so there's little interaction for trees to find, and the explainable model loses nothing.